In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import ObservationWrapper, Wrapper
from gymnasium.spaces import Box


class CropObsWrapper(ObservationWrapper):
    def __init__(self, env, top=34, bottom=194, left=8, right=152):
        super().__init__(env)
        self.top = top
        self.bottom = bottom
        self.left = left
        self.right = right

        # Assuming single channel after AtariWrapper (grayscale)
        shape = (self.bottom - self.top, self.right - self.left, 1)
        self.observation_space = Box(low=0, high=255, shape=shape, dtype=np.uint8)

    def observation(self, obs):
        return obs[self.top:self.bottom, self.left:self.right, :]


class LifePenaltyWrapper(Wrapper):
    def __init__(self, env, penalty=-1.0):
        super().__init__(env)
        self.penalty = penalty
        self.lives = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.lives = info.get("lives", 0)
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        new_lives = info.get("lives", self.lives)
        if new_lives < self.lives:
            reward += self.penalty
        self.lives = new_lives

        return obs, reward, terminated, truncated, info

# Now, let's set up the DQN agent with Stable Baselines3

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv
from stable_baselines3.common.atari_wrappers import AtariWrapper

def make_env(seed=None):
    def _init():
        env = gym.make("ALE/Breakout-v5", render_mode="rgb_array", frameskip=4)
        env = AtariWrapper(env, frame_skip=1, clip_reward=True, terminal_on_life_loss=True)
        # env = CropObsWrapper(env, top=34, bottom=194, left=8, right=152)  # Adjust crop as needed
        # env = LifePenaltyWrapper(env, penalty=-1.0)
        env.reset(seed=seed)
        return env
    return _init

# Parallel environments with proper wrappers
env = DummyVecEnv([make_env() for i in range(8)])

# Frame stacking is crucial for Atari games to give the agent
# a sense of motion and history
env = VecFrameStack(env, n_stack=4)

# Define the DQN model
# MlpPolicy is not suitable for image observations.
# Use CnnPolicy for image-based environments like Atari.
model = DQN(
    "CnnPolicy",
    env,
    learning_rate=1e-4,           # slightly higher LR
    buffer_size=80_000,
    learning_starts=10_000,
    batch_size=32,
    train_freq=4,                # learn every 4 steps
    target_update_interval=10000, # faster target net updates
    exploration_fraction=0.1,    # faster decay
    exploration_initial_eps=1.0,
    exploration_final_eps=0.1,
    verbose=False,
    tensorboard_log="./logs/",
)

# Profile the model
from torchinfo import summary
import torch

# Reset and extract the first observation from the batched env
obs = env.reset()
obs_tensor = torch.tensor(obs[0])                # Shape: [84, 84, 4]
obs_tensor = obs_tensor.permute(2, 0, 1)         # Now: [4, 84, 84]
obs_tensor = obs_tensor.unsqueeze(0).float()     # Now: [1, 4, 84, 84]
obs_tensor = obs_tensor.to(model.device)

# Profile the model
summary(model.policy.q_net, input_data=obs_tensor)